In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
nalisha_tesla_ea_deliveries_and_production_data20152025_path = kagglehub.dataset_download('nalisha/tesla-ea-deliveries-and-production-data20152025')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/nalisha/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv")

In [ ]:
df.head()

# **Data Preprocessing**

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['Model'].value_counts()

In [ ]:
df['Region'].value_counts()

# **Exploratory Data Analysis (EDA)**

In [ ]:
import matplotlib.pyplot as plt

avg_deliveries = df.groupby('Model')['Estimated_Deliveries'].mean()

avg_deliveries.plot(kind='bar')
plt.ylabel('Average Deliveries')
plt.show()

In [ ]:
df.hist(figsize=(12,8))

In [ ]:
import seaborn as sns
sns.boxplot(df['Estimated_Deliveries'])

In [ ]:
monthly = df.groupby('Month')['Estimated_Deliveries'].sum()
monthly.plot(kind='line')

In [ ]:
df.groupby('Year')['Estimated_Deliveries'].sum()


In [ ]:
df.groupby('Region')['Estimated_Deliveries'].mean()

In [ ]:
df.groupby('Model')['Estimated_Deliveries'].mean()

In [ ]:
df.groupby('Source_Type')['Estimated_Deliveries'].mean()

In [ ]:
corr = df.corr(numeric_only=True)
plt.plot(figsize=(12,8))
sns.heatmap(corr, annot=True)
plt.show()

In [ ]:
sns.scatterplot(
    x = 'Production_Units',
    y = 'Estimated_Deliveries',
    data = df
)

In [ ]:
sns.scatterplot(
    x = 'Avg_Price_USD',
    y = 'Estimated_Deliveries',
    data = df
)

In [ ]:
sns.scatterplot(
    x = 'Range_km',
    y = 'Estimated_Deliveries',
    data = df
)

In [ ]:
sns.scatterplot(
    x = 'Battery_Capacity_kWh',
    y = 'Range_km',
    data = df
)

In [ ]:
sns.scatterplot(
    x = 'Charging_Stations',
    y = 'Estimated_Deliveries',
    data = df
)

In [ ]:
sns.scatterplot(
    x = 'CO2_Saved_tons',
    y = 'Estimated_Deliveries',
    data = df
)

# **Feature Engineering**

In [ ]:
df_check = df.copy()

In [ ]:
df_check['Value_score'] = df['Range_km'] / df['Avg_Price_USD']
df_check['Value_score']

In [ ]:
df_check['range_per_kwh'] = df['Range_km'] / df['Battery_Capacity_kWh']
df_check['range_per_kwh']

# **Encoding Categorical Values**

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=['Region','Model','Source_Type'],
    drop_first = True
)

In [ ]:
X = df_encoded.drop('Estimated_Deliveries',axis=1)
y = df['Estimated_Deliveries']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=2)


# **Linear Regression**

In [ ]:
df_new = df_encoded.drop('CO2_Saved_tons',axis=1)
X_new = df_new.drop('Estimated_Deliveries',axis=1)
from sklearn.model_selection import train_test_split
X_new_train, X_new_test, y_new_train, y_new_test = train_test_split(X_new, y, test_size = 0.2, random_state=2)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerical_cols = ['Year',
    'Month',
    'Production_Units',
    'Avg_Price_USD',
    'Battery_Capacity_kWh',
    'Range_km',
    'Charging_Stations']
X_new_train[numerical_cols] = scaler.fit_transform(X_new_train[numerical_cols])
X_new_test[numerical_cols] = scaler.transform(X_new_test[numerical_cols])

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_new_train, y_train)
y_pred = lr.predict(X_new_test)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2_score = r2_score(y_test, y_pred)

In [ ]:
print("mean_absolute_error:", mae)
print("mean_squared_error:",mse)
print("root_mean_squared_error:",np.sqrt(mse))
print("r2_score:", r2_score)

# **Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X_train, y_train)
y_rand_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
rand_mae = mean_absolute_error(y_test, y_rand_pred)
rand_mse = mean_squared_error(y_test, y_rand_pred)
r2_rand_score = r2_score(y_test, y_rand_pred)

In [ ]:
print("Random Forest mean_absolute_error:", rand_mae)
print("Random Forest mean_squared_error:",rand_mse)
print("Random Forest root_mean_squared_error:",np.sqrt(rand_mse))
print("Random Forest r2_score:", r2_rand_score)

# **Hyperparameter Tuning**

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

params = {
    'n_estimators': [100, 200],
    'max_depth': [10, None],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_params_)

# **Cross Validation Scores**

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring='r2'
)

print("CV Scores:", scores)
print("Mean CV R²:", scores.mean())

# **Time Series Forecasting**

In [ ]:
df["Date"] = pd.to_datetime(
    df["Year"].astype(str)
    + "-"
    + df["Month"].astype(str)
)

monthly_sales = (
    df.groupby("Date")["Estimated_Deliveries"]
      .sum()
)

plt.figure(figsize=(12,5))

plt.plot(
    monthly_sales.index,
    monthly_sales.values
)

plt.title("Tesla Deliveries Over Time")

plt.xlabel("Date")

plt.ylabel("Estimated Deliveries")

plt.show()

In [ ]:
import joblib

joblib.dump(model, "tesla_delivery_model.pkl")

print("Model Saved Successfully")